In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# CatBoost

In [2]:
blocks = gpd.read_parquet('../data/traning_data/prepared_blocks.parquet')
blocks.head()

,geometry,residential,business,recreation,industrial,transport,special,agriculture,land_use,share,...,cluster,fsi,gsi,mxi,l,osr,share_living,share_non_living,morphotype,area_accessibility
0,"POLYGON ((352083.617 6633950.146, 352240.448 6...",0.099000,0.0,0.079912,0.000000,0.401072,0.0,0.417018,AGRICULTURE,0.417018,...,3.0,0.000503,0.000503,0.000000,1.000000,1985.451134,0.000000,1.000000,low-rise non-residential,113.847998
1,"POLYGON ((346700.642 6618453.176, 346681.107 6...",1.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,RESIDENTIAL,1.000000,...,2.0,0.064120,0.061465,0.687302,1.043202,14.637095,0.716995,0.326207,individual residential,145.884534
2,"POLYGON ((347043.363 6618261.219, 347042.608 6...",0.729125,0.0,0.270875,0.000000,0.000000,0.0,0.000000,RESIDENTIAL,0.729125,...,2.0,0.034748,0.033472,0.693362,1.038117,27.815100,0.719791,0.318326,individual residential,148.543184
3,"POLYGON ((354879.039 6618859.116, 354845.405 6...",0.454375,0.0,0.000000,0.000000,0.144935,0.0,0.399984,RESIDENTIAL,0.454375,...,2.0,0.183930,0.078846,0.667715,2.332762,5.008184,1.557620,0.775142,low-rise model,132.691062
4,"POLYGON ((347215.933 6646341.789, 347245.429 6...",0.108707,0.0,0.000000,0.767131,0.057528,0.0,0.000000,INDUSTRIAL,0.767131,...,5.0,1.000310,0.250078,0.000000,4.000000,0.749690,0.000000,4.000000,mid-rise non-residential,99.160727


In [3]:
# cols_to_drop = [col for col in blocks.columns if col.startswith('capacity')]
# blocks = blocks.drop(columns=cols_to_drop)
# cols_to_drop = [col for col in blocks.columns if col.startswith('count')]
# blocks = blocks.drop(columns='cluster')
# blocks = blocks.drop(columns=cols_to_drop)

# blocks.columns

In [4]:
from urbanomy.methods.land_value_modeling.training_utils import TrainingConfig, run_training
import logging


feature_cols = [
'residential','business','recreation','industrial','transport','special',
'agriculture','land_use','share','footprint_area','build_floor_area',
'living_area','non_living_area','population','site_area','fsi','gsi',
'mxi','l','morphotype','area_accessibility'
]
cat_features = ['land_use', 'morphotype']
numeric_feats = [c for c in feature_cols if c not in cat_features]

target_col = 'log_total_price'
radius_list = [300, 500, 1000, 2000, 3000]


cfg = TrainingConfig(
    feature_cols=feature_cols,
    cat_features=cat_features,
    radius_list=radius_list,
    target_col=target_col,
    hpo_iter=24,           
    n_clusters=10,
    inner_splits=5,
    outer_splits=5,
    iterations=1500,
    od_wait=300,
    seed=42,
)

model, metrics = run_training(
    blocks,
    cfg,
    log_path="catboost_info/training.log",   # or another path
    console_level=logging.WARNING,           # INFO for more detail, WARNING for minimal console chatter
)
print(metrics)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/mvin/Code/Urbanomy/examples/land_value_modeling/catboost_info/training.log'

In [ ]:
from catboost import Pool
from urbanomy.methods.land_value_modeling.training_utils import spatial_groups, build_lags, prep_cat, feature_names
from sklearn.model_selection import GroupKFold
import shap

groups = spatial_groups(blocks, cfg.n_clusters, random_state=cfg.seed)
_, test_idx = list(GroupKFold(n_splits=cfg.outer_splits).split(blocks, blocks[target_col], groups))[0]
df_test = blocks.iloc[test_idx].copy()

numeric_feats = [c for c in feature_cols if c not in cat_features]
df_test_lag = prep_cat(build_lags(df_test, cfg.radius_list, numeric_feats), cat_features)
feats = feature_names(df_test_lag, feature_cols, target_col)

sample_df = df_test_lag[feats].sample(n=min(1500, len(df_test_lag)), random_state=42)
sample_pool = Pool(sample_df, cat_features=cat_features, feature_names=feats)

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(sample_pool, check_additivity=False)
shap.summary_plot(shap_vals, sample_df, plot_type="bar")
shap.summary_plot(shap_vals, sample_df)

In [ ]:
# final_model.save_model('./data/catboost_model_4_12.cbm')